# VERSION_CHECK — library versions for Table 4.1

**Purpose.** Record the library versions behind the final run, for Table 4.1 of the thesis. It works in two ways:

1. **Original-run evidence**: reads the saved outputs of the locked notebooks (FINAL_00–FINAL_07, AUX_01) and extracts any version that pip printed during the real runs (`Successfully installed …`, `Requirement already satisfied … (x.y.z)`).
2. **Replay**: extracts every `pip install` command from those notebooks, runs the same commands here, and records the resulting versions.

It then checks whether this runtime matches what the final run recorded (Python 3.13.15, scikit-learn 1.6.1).

**How to run.** Use a fresh Colab runtime (Runtime → Disconnect and delete runtime), then Runtime → Run all. Send back the CSV saved at the end. Nothing in the locked folder is modified; only the new `VERSION_CHECK_*.csv/json` files are written.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import glob, os, json, re, sys, shlex, subprocess, platform, datetime
# Set this manually only if auto-detection fails, e.g. '/content/drive/MyDrive/<path>/Implementation'
NOTEBOOK_DIR = None
RECORDED = {'python': '3.13.15', 'scikit-learn': '1.6.1'}   # recorded by FINAL_00 in the final run

if NOTEBOOK_DIR is None:
    hits = glob.glob('/content/drive/MyDrive/**/FINAL_00_Config_and_Freeze.ipynb', recursive=True)
    hits = [h for h in hits if 'SUPERSEDED' not in h]
    assert hits, 'FINAL_00 notebook not found: set NOTEBOOK_DIR manually'
    NOTEBOOK_DIR = os.path.dirname(hits[0])
print('Notebook folder:', NOTEBOOK_DIR)

nbs = sorted(f for f in glob.glob(os.path.join(NOTEBOOK_DIR, '*.ipynb'))
             if re.match(r'(FINAL_0\d|AUX_01)', os.path.basename(f))
             and 'SUPERSEDED' not in f and 'v1' not in os.path.basename(f).split('_')[-1])
for f in nbs: print('  ', os.path.basename(f))
assert len(nbs) >= 9, f'expected 9 locked notebooks, found {len(nbs)}'

Notebook folder: /content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Final_Thesis/Experiments
   AUX_01_SarcOji_Proxy_Validation.ipynb
   FINAL_00_Config_and_Freeze.ipynb
   FINAL_00_Config_and_Freeze_FIXED.ipynb
   FINAL_01_Data_Preparation_and_Subgroups.ipynb
   FINAL_02_Model_Development_and_Lock.ipynb
   FINAL_03_RQ1_Subgroup_Performance.ipynb
   FINAL_04_RQ2_Disparity_Audit.ipynb
   FINAL_05_RQ3_Errors_Confidence_XAI.ipynb
   FINAL_06_RQ4_External_Robustness.ipynb
   FINAL_06_v2_RQ4_External_Robustness.ipynb
   FINAL_07_Ethical_Risk_Synthesis.ipynb


In [3]:
PIP_LINE = re.compile(r'^\s*[!%]\s*pip3?\s+(?:-q\s+)?install\s+(.*)$')
SUBPROC  = re.compile(r"pip['\"]\s*,\s*['\"]install['\"]\s*,(.*?)\]")
OK_INST  = re.compile(r'Successfully installed (.+)')
ALREADY  = re.compile(r'Requirement already satisfied: ([A-Za-z0-9_.\-]+)[^\n]*?\(([0-9][^)\s]*)\)')
installs, evidence, pip_lines_seen = [], {}, []

for f in nbs:
    nb = json.load(open(f))
    name = os.path.basename(f)
    for cell in nb.get('cells', []):
        if cell.get('cell_type') != 'code':
            continue
        src = ''.join(cell.get('source', []))
        for line in src.splitlines():
            if 'pip' in line and 'install' in line:
                pip_lines_seen.append((name, line.strip()))
            m = PIP_LINE.match(line)
            if m:
                installs.append((name, m.group(1).strip()))
        for m in SUBPROC.finditer(src):
            args = re.findall(r"['\"]([^'\"]+)['\"]", m.group(1))
            if args: installs.append((name, ' '.join(args)))
        for out in cell.get('outputs', []):
            text = ''.join(out.get('text', [])) if 'text' in out else ''.join(out.get('data', {}).get('text/plain', []))
            for m in OK_INST.finditer(text):
                for tok in m.group(1).split():
                    pkg, _, ver = tok.rpartition('-')
                    if pkg: evidence.setdefault(pkg.lower(), set()).add((ver, name, 'installed in run'))
            for m in ALREADY.finditer(text):
                evidence.setdefault(m.group(1).lower(), set()).add((m.group(2), name, 'already present in run'))

print('pip lines found in the locked notebooks:')
for n, l in pip_lines_seen: print(f'  [{n}] {l}')
print('\nVersion evidence in saved outputs of the original runs:')
if evidence:
    for k in sorted(evidence): print(' ', k, sorted(evidence[k]))
else:
    print('  none (outputs were quiet or cleared)')

pip lines found in the locked notebooks:
  [AUX_01_SarcOji_Proxy_Validation.ipynb] !pip install -q emoji vaderSentiment pandas pyarrow scikit-learn scipy google-api-python-client
  [FINAL_00_Config_and_Freeze.ipynb] !pip install -q emoji vaderSentiment pandas pyarrow scikit-learn scipy google-api-python-client
  [FINAL_00_Config_and_Freeze_FIXED.ipynb] !pip install -q emoji vaderSentiment pandas pyarrow scikit-learn scipy google-api-python-client
  [FINAL_01_Data_Preparation_and_Subgroups.ipynb] !pip install -q emoji vaderSentiment pandas pyarrow scikit-learn scipy google-api-python-client
  [FINAL_02_Model_Development_and_Lock.ipynb] !pip install -q emoji vaderSentiment pandas pyarrow scikit-learn scipy google-api-python-client
  [FINAL_02_Model_Development_and_Lock.ipynb] !pip install -q xgboost
  [FINAL_03_RQ1_Subgroup_Performance.ipynb] !pip install -q emoji vaderSentiment pandas pyarrow scikit-learn scipy google-api-python-client
  [FINAL_04_RQ2_Disparity_Audit.ipynb] !pip install

In [4]:
# Replay the exact install commands, de-duplicated, in the order they appear
done = []
for name, args in installs:
    if args in done:
        continue
    done.append(args)
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + [a for a in shlex.split(args) if a not in ('-q', '--quiet')]
    print('$', ' '.join(cmd[3:]), f'   (from {name})')
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-800:])

$ install -q emoji vaderSentiment pandas pyarrow scikit-learn scipy google-api-python-client    (from AUX_01_SarcOji_Proxy_Validation.ipynb)
$ install -q xgboost    (from FINAL_02_Model_Development_and_Lock.ipynb)
$ install -q fairlearn aif360    (from FINAL_04_RQ2_Disparity_Audit.ipynb)
$ install -q shap lime    (from FINAL_05_RQ3_Errors_Confidence_XAI.ipynb)


In [5]:
from importlib import metadata
PACKAGES = ['pandas', 'numpy', 'scipy', 'pyarrow', 'scikit-learn', 'xgboost', 'aif360', 'fairlearn',
            'shap', 'lime', 'emoji', 'vaderSentiment', 'matplotlib']
rows = [{'package': 'python', 'replay_version': platform.python_version(),
         'original_run_evidence': RECORDED['python'] + ' (FINAL_00 manifest)'}]
for p in PACKAGES:
    try:
        v = metadata.version(p)
    except metadata.PackageNotFoundError:
        v = 'not installed'
    ev = evidence.get(p.lower()) or evidence.get(p.lower().replace('-', '_')) or set()
    if p == 'scikit-learn':
        ev = set(ev) | {(RECORDED['scikit-learn'], 'FINAL_00', 'manifest')}
    rows.append({'package': p, 'replay_version': v,
                 'original_run_evidence': '; '.join(sorted({f'{e[0]} ({e[1]}, {e[2]})' for e in ev})) or ''})

py_ok = platform.python_version() == RECORDED['python']
sk_ok = metadata.version('scikit-learn') == RECORDED['scikit-learn']
print(f"Python {platform.python_version()}  matches final run: {py_ok}")
print(f"scikit-learn {metadata.version('scikit-learn')}  matches final run: {sk_ok}\n")
print(f"{'package':16s} {'replay version':16s} original-run evidence")
for r in rows:
    print(f"{r['package']:16s} {r['replay_version']:16s} {r['original_run_evidence']}")

Python 3.13.16  matches final run: False
scikit-learn 1.6.1  matches final run: True

package          replay version   original-run evidence
python           3.13.16          3.13.15 (FINAL_00 manifest)
pandas           2.2.3            
numpy            2.1.3            
scipy            1.16.3           
pyarrow          23.0.1           
scikit-learn     1.6.1            1.6.1 (FINAL_00, manifest)
xgboost          3.4.1            
aif360           0.6.1            
fairlearn        0.15.0           
shap             0.52.0           
lime             0.2.0.1          
emoji            2.16.0           
vaderSentiment   3.3.2            
matplotlib       3.10.0           


In [6]:
import csv
stamp = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
meta = {'run_utc': stamp, 'platform': platform.platform(), 'python': platform.python_version(),
        'matches_recorded_python': py_ok, 'matches_recorded_sklearn': sk_ok,
        'notebooks_scanned': [os.path.basename(f) for f in nbs], 'install_commands_replayed': done}
csv_path = os.path.join(NOTEBOOK_DIR, f'VERSION_CHECK_{stamp}.csv')
with open(csv_path, 'w', newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=['package', 'replay_version', 'original_run_evidence'])
    w.writeheader(); w.writerows(rows)
json.dump(meta, open(csv_path.replace('.csv', '.json'), 'w'), indent=2)
print('Saved:', csv_path)
print('Saved:', csv_path.replace('.csv', '.json'))

Saved: /content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Final_Thesis/Experiments/VERSION_CHECK_20261012_091508.csv
Saved: /content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Final_Thesis/Experiments/VERSION_CHECK_20261012_091508.json


/tmp/ipykernel_4742/2268849739.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  stamp = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
